In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
SEASONAL_WEEKS = 52
TEST_YEAR = 2025
FORECAST_HORIZON = 8

PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / "data" / "processed"
RAW = PROJECT_ROOT / "data" / "raw"

cleaned_path = DATA / "cleaned_sales.csv"
if not cleaned_path.exists():
    raise FileNotFoundError(f"Missing {cleaned_path}. Run D1 first.")

df = pd.read_csv(cleaned_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "sku_id", "quantity"]).copy()
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").fillna(0).clip(lower=0)
print("Loaded cleaned sales:", df.shape)

/home/oai/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-ck7th9pk because there was an issue with the default path (/home/oai/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Loaded cleaned sales: (99863, 11)


In [2]:
# Weekly SKU demand — one row per SKU/week
df["week"] = df["date"].dt.to_period("W-MON").apply(lambda p: p.start_time)
weekly = (df.groupby(["week", "sku_id"], as_index=False)["quantity"].sum()
            .rename(columns={"quantity":"weekly_quantity"}))

# Complete the SKU x week grid so zero-sales weeks are explicit.
all_weeks = pd.date_range(weekly["week"].min(), weekly["week"].max(), freq="7D")
all_skus = weekly["sku_id"].dropna().unique()
idx = pd.MultiIndex.from_product([all_weeks, all_skus], names=["week","sku_id"])
weekly = (weekly.set_index(["week","sku_id"]).reindex(idx, fill_value=0).reset_index())
weekly["weekly_quantity"] = pd.to_numeric(weekly["weekly_quantity"], errors="coerce").fillna(0)
print("Weekly dataset:", weekly.shape, "SKUs:", weekly.sku_id.nunique())

Weekly dataset: (1049580, 3) SKUs: 4998


In [3]:
# Time-aware feature engineering. Every lag/rolling feature uses only observations at or before the current week.
g = weekly.sort_values(["sku_id","week"]).copy()
g["lag_1"] = g.groupby("sku_id")["weekly_quantity"].shift(1)
g["lag_2"] = g.groupby("sku_id")["weekly_quantity"].shift(2)
g["lag_4"] = g.groupby("sku_id")["weekly_quantity"].shift(4)
g["lag_8"] = g.groupby("sku_id")["weekly_quantity"].shift(8)
g["lag_52"] = g.groupby("sku_id")["weekly_quantity"].shift(52)
# Shift before rolling so the current target is never included.
g["rolling_mean_4"] = g.groupby("sku_id")["weekly_quantity"].transform(lambda s: s.shift(1).rolling(4, min_periods=4).mean())
g["rolling_mean_8"] = g.groupby("sku_id")["weekly_quantity"].transform(lambda s: s.shift(1).rolling(8, min_periods=8).mean())
g["week_of_year"] = g["week"].dt.isocalendar().week.astype(int)
g["month"] = g["week"].dt.month
g["quarter"] = g["week"].dt.quarter
g["season"] = g["month"].map({12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",6:"Summer",7:"Summer",8:"Summer",9:"Autumn",10:"Autumn",11:"Autumn"})

feature_cols = ["lag_1","lag_2","lag_4","lag_8","lag_52","rolling_mean_4","rolling_mean_8","week_of_year","month","quarter","season"]
target_col = "weekly_quantity"
model_df = g.dropna(subset=["lag_52","rolling_mean_4","rolling_mean_8"]).copy()
print("Model rows:", len(model_df))

Model rows: 789684


In [4]:
# Correct chronological holdout: all pre-2025 observations train, 2025 observations test.
train = model_df[model_df["week"].dt.year < TEST_YEAR].copy()
test = model_df[model_df["week"].dt.year == TEST_YEAR].copy()
assert not train.empty and not test.empty
assert train["week"].max() < test["week"].min()

X_train = pd.get_dummies(train[feature_cols], columns=["season"], dtype=float)
X_test = pd.get_dummies(test[feature_cols], columns=["season"], dtype=float).reindex(columns=X_train.columns, fill_value=0)
y_train = train[target_col].astype(float)
y_test = test[target_col].astype(float)

print("Train:", train["week"].min(), "to", train["week"].max(), len(train))
print("Test :", test["week"].min(), "to", test["week"].max(), len(test))
print("Chronological split: PASS")

Train: 2022-12-27 00:00:00 to 2024-12-31 00:00:00 529788
Test : 2025-01-07 00:00:00 to 2025-12-30 00:00:00 259896
Chronological split: PASS


In [5]:
# Final Random Forest model
model = RandomForestRegressor(n_estimators=30, random_state=SEED, n_jobs=-1, min_samples_leaf=2, max_samples=0.30)
model.fit(X_train, y_train)
y_pred = np.maximum(model.predict(X_test), 0)

def wape(y_true, y_pred):
    denom = np.sum(np.abs(np.asarray(y_true)))
    return np.nan if denom == 0 else np.sum(np.abs(np.asarray(y_true)-np.asarray(y_pred))) / denom * 100

model_wape = wape(y_test, y_pred)
model_mae = mean_absolute_error(y_test, y_pred)
model_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
model_bias = float(np.mean(y_pred - y_test.to_numpy()))
print(f"Model WAPE: {model_wape:.2f}%")
print(f"MAE: {model_mae:.4f} | RMSE: {model_rmse:.4f} | Bias: {model_bias:.4f}")

Model WAPE: 155.05%
MAE: 0.2945 | RMSE: 0.7345 | Bias: 0.0029


In [6]:
# Same-period seasonal-naive baseline.
# For each SKU/week in the test period, use demand exactly 52 weeks earlier.
lookup = weekly.set_index(["sku_id","week"])["weekly_quantity"]
test["baseline_forecast"] = [lookup.get((sku, week - pd.Timedelta(weeks=SEASONAL_WEEKS)), np.nan) for sku, week in zip(test.sku_id, test.week)]
valid = test["baseline_forecast"].notna()
comparison = test.loc[valid, ["sku_id","week","weekly_quantity","baseline_forecast"]].copy()
comparison["model_forecast"] = y_pred[valid.to_numpy()]
baseline_wape = wape(comparison["weekly_quantity"], comparison["baseline_forecast"])
model_wape_fair = wape(comparison["weekly_quantity"], comparison["model_forecast"])
print(f"Seasonal-naive WAPE: {baseline_wape:.2f}%")
print(f"Model WAPE (same rows): {model_wape_fair:.2f}%")
print(f"Improvement: {baseline_wape-model_wape_fair:.2f} percentage points")

Seasonal-naive WAPE: 156.65%
Model WAPE (same rows): 155.05%
Improvement: 1.60 percentage points


In [7]:
# Rolling-origin CV on the pre-2025 training history only.
# Split by unique weeks, not arbitrary rows, so every fold is genuinely past -> future.
cv_base = train.sort_values("week").copy()
unique_weeks = np.array(sorted(cv_base["week"].unique()))
if len(unique_weeks) < 10:
    raise ValueError("Not enough historical weeks for rolling-origin CV.")

n_splits = min(5, len(unique_weeks)//2)
tscv = TimeSeriesSplit(n_splits=n_splits)
cv_rows=[]
for fold, (tr_idx, te_idx) in enumerate(tscv.split(unique_weeks), 1):
    tr_weeks = set(unique_weeks[tr_idx]); te_weeks = set(unique_weeks[te_idx])
    tr = cv_base[cv_base["week"].isin(tr_weeks)]
    te = cv_base[cv_base["week"].isin(te_weeks)]
    Xtr = pd.get_dummies(tr[feature_cols], columns=["season"], dtype=float)
    Xte = pd.get_dummies(te[feature_cols], columns=["season"], dtype=float).reindex(columns=Xtr.columns, fill_value=0)
    m = RandomForestRegressor(n_estimators=30, random_state=SEED, n_jobs=-1, min_samples_leaf=2, max_samples=0.30)
    m.fit(Xtr, tr[target_col])
    pred = np.maximum(m.predict(Xte),0)
    cv_rows.append({"fold":fold,"train_end":tr.week.max(),"test_start":te.week.min(),"test_end":te.week.max(),"MAE":mean_absolute_error(te[target_col],pred),"RMSE":np.sqrt(mean_squared_error(te[target_col],pred)),"WAPE":wape(te[target_col],pred)})

cv_results = pd.DataFrame(cv_rows)
print(cv_results)
print("Mean rolling WAPE:", round(cv_results["WAPE"].mean(),2), "%")

   fold  train_end test_start   test_end       MAE      RMSE        WAPE
0     1 2023-05-16 2023-05-23 2023-09-12  0.286724  0.725847  167.884944
1     2 2023-09-12 2023-09-19 2024-01-09  0.320728  0.753783  158.823624
2     3 2024-01-09 2024-01-16 2024-05-07  0.263839  0.682150  157.646846
3     4 2024-05-07 2024-05-14 2024-09-03  0.276884  0.704867  155.572707
4     5 2024-09-03 2024-09-10 2024-12-31  0.324904  0.792996  153.476551
Mean rolling WAPE: 158.68 %


In [8]:
# Leakage audit
assert train.week.max() < test.week.min()
for col in ["lag_1","lag_2","lag_4","lag_8","lag_52","rolling_mean_4","rolling_mean_8"]:
    assert col in feature_cols
# Rolling features were explicitly shifted(1), so current target is excluded.
assert not model_df[feature_cols].isna().any().any()
print("Leakage audit: PASS")

Leakage audit: PASS


In [9]:
# 8-week forward forecast for every SKU with sufficient history.
# Recursive prediction: each next week uses only observed history + prior model predictions.
history = weekly[["sku_id","week","weekly_quantity"]].copy().sort_values(["sku_id","week"])
last_week = history["week"].max()
skus = history["sku_id"].unique()
future_weeks = [last_week + pd.Timedelta(weeks=i) for i in range(1, FORECAST_HORIZON+1)]

future_rows=[]
series={sku: list(zip(h.week.tolist(), h.weekly_quantity.astype(float).tolist())) for sku,h in history.groupby("sku_id")}
for week in future_weeks:
    for sku in skus:
        vals = {w:v for w,v in series[sku]}
        def lag(n): return vals.get(week-pd.Timedelta(weeks=n), np.nan)
        r4 = [lag(i) for i in range(1,5)]
        r8 = [lag(i) for i in range(1,9)]
        row={"sku_id":sku,"week":week,"lag_1":lag(1),"lag_2":lag(2),"lag_4":lag(4),"lag_8":lag(8),"lag_52":lag(52),"rolling_mean_4":np.mean(r4) if all(pd.notna(r4)) else np.nan,"rolling_mean_8":np.mean(r8) if all(pd.notna(r8)) else np.nan,"week_of_year":int(week.isocalendar().week),"month":week.month,"quarter":week.quarter,"season":{12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",6:"Summer",7:"Summer",8:"Summer",9:"Autumn",10:"Autumn",11:"Autumn"}[week.month]}
        future_rows.append(row)
    # Predict after features are constructed for this week.
    step = pd.DataFrame([r for r in future_rows if r["week"]==week])
    usable = step.dropna(subset=["lag_52","rolling_mean_4","rolling_mean_8"]).copy()
    if not usable.empty:
        Xf = pd.get_dummies(usable[feature_cols], columns=["season"], dtype=float).reindex(columns=X_train.columns, fill_value=0)
        pred = np.maximum(model.predict(Xf),0)
        for sku,p in zip(usable.sku_id,pred): series[sku].append((week,float(p)))

future_forecast = pd.DataFrame(future_rows)
# Rebuild predictions from the recursively generated series.
# Safer direct lookup for every row.
forecast_lookup={(sku,w):v for sku,arr in series.items() for w,v in arr}
future_forecast["forecast_demand"]=[forecast_lookup.get((sku,w),np.nan) for sku,w in zip(future_forecast.sku_id,future_forecast.week)]
future_forecast.to_csv(DATA/"future_8_week_forecast.csv", index=False)
print("Future forecast saved:", DATA/"future_8_week_forecast.csv")
print("Rows:",len(future_forecast),"Weeks:",future_forecast.week.nunique(),"SKUs:",future_forecast.sku_id.nunique())

Future forecast saved: /mnt/data/FORESIGHT_TEST_PROJECT/data/processed/future_8_week_forecast.csv
Rows: 39984 Weeks: 8 SKUs: 4998


In [10]:
# Final D3 acceptance check
checks = {
    "Weekly SKU forecast": weekly.groupby(["week","sku_id"]).size().max() == 1,
    "Seasonal-naive baseline": pd.notna(baseline_wape),
    "Rolling-origin CV": len(cv_results) >= 2,
    "WAPE comparison": pd.notna(model_wape_fair) and pd.notna(baseline_wape),
    "No leakage": train.week.max() < test.week.min(),
    "8-week forward forecast": future_forecast.week.nunique() == FORECAST_HORIZON,
}
for k,v in checks.items(): print(f"{k}: {'PASS' if v else 'FAIL'}")
assert all(checks.values())
print("\nD3 FINAL ACCEPTANCE: PASS")

Weekly SKU forecast: PASS
Seasonal-naive baseline: PASS
Rolling-origin CV: PASS
WAPE comparison: PASS
No leakage: PASS
8-week forward forecast: PASS

D3 FINAL ACCEPTANCE: PASS
